# StreamGate-TF: result and state walkthrough

Inspect the executed experiment and use the streaming API. This notebook does not train the models. The checkpoints are small byte-language models, not instruction-following assistants.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import json
from pathlib import Path
metrics = json.loads(Path('metrics.json').read_text())
for kind, result in metrics['models'].items():
    score = result['holdout']
    print(f"{kind:8s} perplexity={score['perplexity']:.4f}, accuracy={score['next_byte_accuracy']:.2%}")

softmax  perplexity=9.0866, accuracy=35.63%
linear   perplexity=6.8628, accuracy=43.89%
unigram  perplexity=23.5051, accuracy=16.94%


## Learning and holdout quality

![Learning curves](figures/learning-curves.png)

![Model comparison](figures/model-comparison.png)

Both architectures saw identical training-window starts and the same number of byte targets. Validation selected checkpoints and architecture; the holdout did not. This is one seed on one small English corpus.

In [2]:
from streamgate.inference import StreamingEngine
engine = StreamingEngine.load(kind='linear')
session = engine.session('Holmes said, ')
initial_payload = session.attention_state_bytes
continuation = session.generate(count=32, seed=42)
print(continuation.decode('utf-8', errors='replace'))
print('Before:', initial_payload, 'After:', session.attention_state_bytes, 'bytes')
assert session.attention_state_bytes == initial_payload == 8704

shougk the who go the with a sow
Before: 8704 After: 8704 bytes


## The measured trade-off

![State memory](figures/state-memory.png)

![CPU latency](figures/streaming-latency.png)

Fixed attention state is not total-process RAM. StreamGate's sequential prefill made whole requests slower than the cached softmax comparator in this CPU workload. A 256-byte prefix tests state mechanics and scaling, not useful quality beyond the 64-byte training context.

In [3]:
import numpy as np
import tensorflow as tf
tokens = tf.constant([[72,111,108,109,101,115]], tf.int32)
full = engine.model(tokens).numpy()
first, state = engine.model.prefill(tokens[:,:3])
second, state = engine.model.prefill(tokens[:,3:], state=state, offset=3)
chunked = np.concatenate([first.numpy(), second.numpy()], axis=1)
print('Maximum chunk/full logit difference:', np.abs(full-chunked).max())
np.testing.assert_allclose(full, chunked, atol=1e-4, rtol=1e-4)

Maximum chunk/full logit difference: 1.9073486e-06


## Errors and gates

![Error categories](figures/error-by-byte-group.png)

![Gate distributions](figures/learned-gates.png)

Review the highest-loss snippets in `runs/error_analysis.json`. Gate differences describe learned retention patterns; they do not prove semantic specialization. Human review prompts in `LEARNING_NOTES.md` are intentionally unfilled.

In [4]:
benchmark = json.loads(Path('runs/benchmark.json').read_text())
verification = json.loads(Path('runs/verification.json').read_text())
assert verification['status'] == 'passed'
assert all(item['all_greedy_predictions_match'] for item in benchmark['parity'])
print('Fresh-source checkpoint replay:', verification['status'])
print('Largest recorded streaming/full logit difference:', max(item['max_absolute_logit_error'] for item in benchmark['parity']))

Fresh-source checkpoint replay: passed
Largest recorded streaming/full logit difference: 3.337860107421875e-06
